- Run the DMPE algorithm with a static objective that is adapated to the dynamics of the system (higher density for higher dynamics).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

---

In [ ]:
from dmpe.utils.density_estimation import build_grid, DensityEstimate
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from admpe.vector_fields import VectorField, DynamicsVectorField, CurvatureVectorField
from admpe.adaptive_densities import TargetDensity

In [ ]:
points_per_dim = 10

env, penalty_function, featurize, env_params = setup_cart_pole_env()
dvf = DynamicsVectorField.from_env(env, points_per_dim, low=-1.05, high=1.05)
dvf.visualize()

In [ ]:
target_density = TargetDensity.from_vector_field(
    vf=dvf,
    bandwidth=0.1,
    penalty_function=penalty_function,
    consider_action_distribution=True,
    obs_dim=env.obs_dim,
    act_dim=env.action_dim,
)

In [ ]:
target_density.visualize(labels=list(env.obs_description) + list(env.action_description))

---

Perform excitation with new target distribution:

(compare to uniform target)

In [ ]:
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.algorithms.algorithm_utils import default_dmpe_parameterization
from dmpe.algorithms.algorithms import excite_with_dmpe

from dmpe.evaluation.plotting_utils import plot_feature_combinations

In [ ]:
exp_params, proposed_actions, loader_key, expl_key = default_dmpe_parameterization(
    env,
    seed=0,
    featurize=featurize,
    model_class=NeuralEulerODECartpole,
    n_time_steps=10_000,
    points_per_dim=points_per_dim,
)
exp_params["alg_params"]["target_distribution"] = target_density.p[0]

observations, actions, model, density_estimate, losses, proposed_actions, callback_out = excite_with_dmpe(
    env,
    exp_params,
    proposed_actions,
    loader_key,
    expl_key,
)

data_exp_adpt = dict(
    observations=observations,
    actions=actions,
    model=model,
    density_estimate=density_estimate,
    losses=losses,
    proposed_actions=proposed_actions,
)

In [ ]:
print("target")
target_density.visualize()
plt.show()

print("result")
data_exp_adpt["density_estimate"].visualize(labels=list(env.obs_description) + list(env.action_description))
plt.show()

plot_feature_combinations(jnp.hstack([
    data_exp_adpt["observations"][:-1],
    data_exp_adpt["actions"]
]), list(env.obs_description) + list(env.action_description));
plt.show()

In [ ]:
exp_params, proposed_actions, loader_key, expl_key = default_dmpe_parameterization(
    env,
    seed=0,
    featurize=featurize,
    model_class=NeuralEulerODECartpole,
    n_time_steps=10_000,
    points_per_dim=points_per_dim,
)

uniform_density = DensityEstimate.from_estimate(p=exp_params["alg_params"]["target_distribution"], n_additional_observations=0, density_estimate=target_density)

observations, actions, model, density_estimate, losses, proposed_actions, callback_out = excite_with_dmpe(
    env,
    exp_params,
    proposed_actions,
    loader_key,
    expl_key,

)

data_exp_uniform = dict(
    observations=observations,
    actions=actions,
    model=model,
    density_estimate=density_estimate,
    losses=losses,
    proposed_actions=proposed_actions,
)

In [ ]:
print("target")
uniform_density.visualize()
plt.show()

print("result")
data_exp_uniform["density_estimate"].visualize(labels=list(env.obs_description) + list(env.action_description))
plt.show()

plot_feature_combinations(jnp.hstack([
    data_exp_uniform["observations"][:-1],
    data_exp_uniform["actions"]
]), list(env.obs_description) + list(env.action_description));
plt.show()